# 25-Gaussian mixture — Anisotropic

## Imports

In [ ]:
import sys

sys.path.append("..")

import math

import numpy as np
import torch
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

from src.schr_bridge import Schr_Bridge
from src.distributions import TruncatedNormalSampler

%matplotlib inline

## Configuration

In [ ]:
DIM = 2
BATCH_SIZE = 128
D_LR = 2e-3
D_GRADIENT_MAX_NORM = float("inf")
MAX_STEPS = 2000

EPSILON = 0.1
N_POTENTIALS = 50
B = 0.801875
MU = -0.019881
T = 10.0
S_DIAGONAL_INIT = 1.1
S_LOWER_BOUND = 1.0
IS_DIAGONAL = True
SAMPLING_BATCH_SIZE = 128
INIT_BY_SAMPLES = True

M_SLACK = 3.0
CONSTRAINT_WEIGHT = 5.0

EVAL_SEED = 0xBADBEEF
FIGURE_NAME = "anisotropic_50.png"

In [ ]:
def setup_consistent_evaluation(seed=EVAL_SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

## Distributions

In [ ]:
def create_target_distributions():
    setup_consistent_evaluation()
    grid_positions = np.linspace(-6, 6, 5)
    means = np.array([(x, y) for x in grid_positions for y in grid_positions])

    covs = []
    for i in range(len(means)):
        theta = np.random.uniform(0, np.pi)
        rot = np.array([[np.cos(theta), -np.sin(theta)],
                        [np.sin(theta), np.cos(theta)]])

        eigvals = np.diag(np.random.uniform(0.05, 0.5, 2))

        cov = rot @ eigvals @ rot.T
        covs.append(cov)

    return means, covs


target_means, target_covs = create_target_distributions()

In [ ]:
class GaussianGridSampler:
    def __init__(self, means, covs):
        self.means = [torch.tensor(m, dtype=torch.float32) for m in means]
        self.covs = [torch.tensor(c, dtype=torch.float32) for c in covs]
        self.n_components = len(means)
        self.dim = len(means[0])

    def sample(self, n):
        component = torch.randint(0, self.n_components, (n,))
        samples = torch.zeros(n, self.dim)
        for i in range(n):
            c = component[i]
            samples[i] = torch.distributions.MultivariateNormal(
                self.means[c],
                covariance_matrix=self.covs[c],
            ).sample()
        return samples


X_sampler = TruncatedNormalSampler(DIM)
Y_sampler = GaussianGridSampler(target_means, target_covs)

In [ ]:
def plot_distributions(target_samples, transformed_samples=None, filename=None,
                       color="tab:orange"):
    plt.figure(figsize=(10, 10))

    plt.scatter(target_samples[:, 0], target_samples[:, 1], color="g",
                alpha=0.3, label="Target (Gaussian mixture)")

    if transformed_samples is not None:
        plt.scatter(transformed_samples[:, 0], transformed_samples[:, 1], color=color,
                    alpha=0.3, label="Transformed")

    for mean in target_means:
        plt.plot(mean[0], mean[1], "ro", markersize=8)

    plt.grid(True)

    if filename:
        plt.savefig(filename)
    plt.show()

In [ ]:
setup_consistent_evaluation()

plot_distributions(Y_sampler.sample(1000).numpy())

## Training

In [ ]:
def train_OU_model(b, mu, n_potentials=N_POTENTIALS):
    D = Schr_Bridge(
        dim=DIM,
        n_potentials=n_potentials,
        epsilon=EPSILON,
        b=b,
        m=mu,
        T=T,
        sampling_batch_size=SAMPLING_BATCH_SIZE,
        S_diagonal_init=S_DIAGONAL_INIT,
        S_lower_bound=S_LOWER_BOUND,
        is_diagonal=IS_DIAGONAL,
    )

    if INIT_BY_SAMPLES:
        D.init_r_by_samples(Y_sampler.sample(n_potentials))

    D_opt = torch.optim.Adam(D.parameters(), lr=D_LR)

    loss_history, grad_norm_history = [], []

    n_steps = max(MAX_STEPS, round(MAX_STEPS * n_potentials / 20))

    for step in tqdm(range(n_steps)):
        D_opt.zero_grad()

        X0, X1 = X_sampler.sample(BATCH_SIZE), Y_sampler.sample(BATCH_SIZE)

        log_potential = D.get_log_potential(X1)
        log_C_tilde = D.get_log_C_tilde(D.T)

        e_sq = math.e ** 2
        K_T = ((1 - torch.exp(-2 * D.b * D.T)) ** (-5 * e_sq * math.sqrt(D.dim))
               * torch.exp(2 * e_sq * math.sqrt(D.dim) * torch.arcsin(torch.exp(-D.b * D.T))))

        sigma_T_sq = D.ou_sigma(D.b, D.T)
        quad = (X1 ** 2).sum(dim=-1) / (2 * D.epsilon * sigma_T_sq)
        phi = log_potential + quad
        penalty = torch.relu(phi - (quad.max() + M_SLACK)).mean()

        D_loss = ((1.0 / K_T) * log_C_tilde - log_potential).mean() + CONSTRAINT_WEIGHT * penalty
        D_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(D.parameters(), max_norm=D_GRADIENT_MAX_NORM)
        D_opt.step()

        loss_history.append(D_loss.item())
        grad_norm_history.append(grad_norm.item())

    return D, loss_history, grad_norm_history

In [ ]:
setup_consistent_evaluation()
D, loss_history, grad_norm_history = train_OU_model(B, MU)

print(f"mean loss over the last 100 steps: {np.mean(loss_history[-100:]):.4f}")

## Visualization

In [ ]:
D.eval()

with torch.no_grad():
    source_samples = X_sampler.sample(10000)
    target_samples = Y_sampler.sample(10000)

    transformed = D(source_samples)

plot_distributions(
    target_samples.numpy(),
    transformed.numpy(),
    filename=FIGURE_NAME,
)